# Lab 1 金融爬蟲：把公開資料變成表格

**今天的目標：** 寫程式自動去網路上把**公開的金融資料**抓回來、整理成表格。你會親手產出兩個檔案：
- **`prices.csv`**（證交所股價）→ 之後 pandas / KNN / 決策樹要用的「數據路」原料
- **`news_raw.csv`**（財經新聞 date/title/text）→ 之後情緒打分要用的「文字路」原料

> 🚫 **免責：** 今天抓的股價、新聞純為教學示範「程式怎麼把公開資料抓回來」，**不是投資建議**。
> 📜 **法遵三句：** ① 只抓公開、不用登入的資料；② 只做課堂教學分析、不轉貼全文、不商用；③ 守禮貌頻率（每抓一頁睡 1 秒、只抓前幾則）。robots.txt「告示牌」怎麼看，老師會講。

In [1]:
# 📦 先跑這一格：一次裝齊今天要用的套件（已裝好的會直接跳過；裝不起來看 README）
!pip install -q requests beautifulsoup4 pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
# 老師的小設定：關掉一個無關緊要的套件提醒，讓等一下的輸出乾淨一點（直接跑、不用改）
import warnings
warnings.filterwarnings("ignore")

## 🔧 第 0 步：環境檢查

**預期輸出：** 印出 `requests` 和 `bs4` 的版本號——有版本號就代表套件裝好了。

In [3]:
import requests, bs4
print("requests", requests.__version__)
print("bs4     ", bs4.__version__)

requests 2.32.5
bs4      4.14.3


---
## A・第一抓：網頁就是一坨文字

`requests.get(網址)` ＝ 寫程式「去跟這個網址要資料」。先把中央社財經新聞的 **RSS**（網站給程式讀的「最新文章清單」）抓回來，看兩件事：① 回來的是一坨文字；② 狀態碼 `200` ＝成功。

> 💬 這種公開清單**不用表明身分也抓得到**——先體會最單純的一行抓網頁（到 C2 抓內頁時，我們才會遇到「要表明身分」的情況）。

**預期輸出：** 狀態碼 `200`，接著一段 `<?xml ...>` 開頭的文字。⚠️ 你抓到的內容**跟這裡不一樣是正常的**——新聞每分鐘在變。

In [4]:
import requests

# RSS 這種公開清單，不帶任何身分也抓得到——最單純的「一行抓網頁」
r = requests.get("https://feeds.feedburner.com/rsscna/finance")
print("狀態碼：", r.status_code)          # 💡 200＝成功、404＝沒這頁、403/429＝被擋
print("--- 回來的內容前 500 個字 ---")
print(r.text[:500])                       # 💡 網頁（這裡是 RSS）真的就是一坨文字，一行就拿到了

狀態碼： 200
--- 回來的內容前 500 個字 ---
<?xml version="1.0" encoding="UTF-8" ?>
<rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom" xmlns:content="http://purl.org/rss/1.0/modules/content/">
  <channel>
     <title>中央社即時新聞 財經新聞</title>
     <link>https://www.cna.com.tw/list/aie.aspx</link>
     <description>中央社即時新聞 財經新聞</description>
     <language>zh-tw</language>
     <copyright>中央通訊社</copyright>
     <lastBuildDate>Sun, 19 Jul 2026 00:30:11 +0800</lastBuildDate>
     <ttl>15</ttl>
     <image>
       <title>中央社即時新


### A・順便認識一次「出錯長相」

故意去要一個不存在的網址，看 `404` 長什麼樣。常見狀態碼：**200** 成功、**404** 沒這頁、**403 / 429** 被擋（老師會多講幾個）。

**預期輸出：** `404`。

In [5]:
# 把剛剛那個 RSS 網址故意打錯（feedburner 上不存在的路徑）
bad = requests.get("https://feeds.feedburner.com/rsscna/nonexistent_xyz")
print("故意打錯網址的狀態碼：", bad.status_code)   # 💡 出錯訊息是爬蟲工程師的日常，先跟它打個照面

故意打錯網址的狀態碼： 404


### 📝 小作業 A

1. 把 `r.text[:500]` 改成 `r.text[:1500]`，多看一點——**數數看你看到了幾個 `<item>`？**（每個 `<item>` 就是一則文章）

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
print(r.text[:1500])
# 整份 RSS 就是一串 <item> 排下來，每個 <item> 裡有 <title>（標題）和 <link>（內頁網址）。
```
**結論：** RSS 是「一串 `<item>` 清單」——C2 段就是靠它拿到「有哪些文章 + 網址」。
</details>

In [6]:
# 📝 小作業 A 參考解（參考解不只一種，思路對就好）

# 多看一點，數數看有幾個 <item>
print(r.text[:1500])
# 💡 整份 RSS 就是一串 <item> 排下來；每個 <item> 裡有 <title> 和 <link>

<?xml version="1.0" encoding="UTF-8" ?>
<rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom" xmlns:content="http://purl.org/rss/1.0/modules/content/">
  <channel>
     <title>中央社即時新聞 財經新聞</title>
     <link>https://www.cna.com.tw/list/aie.aspx</link>
     <description>中央社即時新聞 財經新聞</description>
     <language>zh-tw</language>
     <copyright>中央通訊社</copyright>
     <lastBuildDate>Sun, 19 Jul 2026 00:30:11 +0800</lastBuildDate>
     <ttl>15</ttl>
     <image>
       <title>中央社即時新聞 財經新聞</title>
       <width>114</width>
       <height>67</height>
       <link>https://www.cna.com.tw/list/aie.aspx</link>
       <url>https://imgcdn.cna.com.tw/www/images/cna_info/cnalogo_176x117.jpg</url>
     </image>
     <item>
     <title>今彩539第115174期　頭獎1注中獎</title>
     <link>https://www.cna.com.tw/news/ahel/202607180215.aspx</link>
     <guid isPermaLink="false">CNA/2026-07-18/202607180215</guid>
     <pubDate>Sat, 18 Jul 2026 22:04:07 +0800</pubDate>
     <description><![CDATA[（中央社台北18日電）今彩539第

---
## B・數據路：證交所 API 抓股價 → `prices.csv`

抓股價我們**不爬網頁、直接打 API**——證交所（TWSE）是政府公開資料，提供官方查詢 API，格式乾淨、本來就歡迎程式來拿。**有 API 就用 API、別硬爬**，這是爬蟲第一守則。

> 💬 `params` ＝ 把查詢條件（要哪檔、哪個月）掛到網址後面；`r.json()` ＝ 把回應那串 JSON 文字轉成你熟的 dict。

**預期輸出：** `stat： OK`、10 個欄位名、第一個交易日那列。⚠️ 行情數字是當日快照、**你跑的一定不同**。

In [7]:
import requests

# 證交所「個股日成交資訊」API：date=查詢月份、stockNo=股票代號（2330 只是查詢參數）
r = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                 params={"response": "json", "date": "20260701", "stockNo": "2330"})
j = r.json()                      # 💡 JSON → dict，一行；之後就是你熟的 j["鍵"] 取值
print("stat：", j["stat"])        # "OK" = 查詢成功
print("欄位：", j["fields"])
print("第一列：", j["data"][0])   # 💡 j["data"] 是一列一列的資料，[0]＝第一個交易日

stat： OK
欄位： ['日期', '成交股數', '成交金額', '開盤價', '最高價', '最低價', '收盤價', '漲跌價差', '成交筆數', '註記']
第一列： ['115/07/01', '37,544,470', '93,600,076,825', '2,495.00', '2,505.00', '2,475.00', '2,505.00', '+95.00', '111,091', '']


### B・變成表格

`DataFrame` ＝ pandas 幫你把資料組成「像 Excel 的表格」。

**預期輸出：** 一張表格（前幾列）。

In [8]:
import pandas as pd

df_price = pd.DataFrame(j["data"], columns=j["fields"])   # 💡 用 fields 當欄名、data 當內容，組成表格
print(df_price.head())

          日期        成交股數            成交金額       開盤價       最高價       最低價  \
0  115/07/01  37,544,470  93,600,076,825  2,495.00  2,505.00  2,475.00   
1  115/07/02  35,919,290  88,369,879,773  2,450.00  2,480.00  2,445.00   
2  115/07/03  32,905,868  80,082,636,340  2,415.00  2,465.00  2,415.00   
3  115/07/06  21,041,918  52,022,654,454  2,465.00  2,500.00  2,455.00   
4  115/07/07  31,400,854  77,617,188,273  2,480.00  2,500.00  2,440.00   

        收盤價    漲跌價差     成交筆數 註記  
0  2,505.00  +95.00  111,091     
1  2,465.00  -40.00  132,697     
2  2,445.00  -20.00  154,750     
3  2,460.00  +15.00   80,246     
4  2,440.00  -20.00  132,943     


### B・存成 `prices.csv`

`df` 可以匯出成很多格式（`.csv`、`.xlsx`…）；最通用的是 `.csv`。這個檔就是後面 pandas / KNN / 決策樹 的原料。

**預期輸出：**「已存 prices.csv」。

In [9]:
print("共", len(df_price), "個交易日")
df_price.to_csv("prices.csv", index=False)
print("已存 prices.csv（數據路的原料）")

共 12 個交易日
已存 prices.csv（數據路的原料）


### B・取值小知識：`.iloc`

`.iloc[0]` ＝ 用「第幾列」的位置取值（`0` ＝第一列）。等一下用它抓出兩個「髒資料」給你看。

### B・當眾抓兩個「髒資料」（模組 2 的鉤子）

真實世界抓回來的資料**永遠是髒的**。指出兩個等一下 pandas 要修的地方。

**預期輸出：** 一個民國日期字串、一個含逗號的字串、型別是 `str`。

In [10]:
print(df_price["日期"].iloc[0])       # 💡 '115/06/01' ← 證交所用民國紀年（115 年＝西元 2026 年），要修
print(df_price["成交股數"].iloc[0])   # 💡 '60,942,792' ← 含千分位逗號，其實是字串不是數字，要修
print(type(df_price["成交股數"].iloc[0]))   # 💡 <class 'str'>——直接拿去算平均會爆錯
# 今天先不修！洗乾淨（民國轉西元、逗號轉數字）正是模組 2 pandas 要教的第一件事。

115/07/01
37,544,470
<class 'str'>


### 📝 小作業 B

1. 把 `stockNo` 換成別檔（例如 `"2317"` 鴻海），重跑抓成表格——**整段解析 code 一行都不用動**（這就是 API 的好處：條件是參數）。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
r2 = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                  params={"response": "json", "date": "20260701", "stockNo": "2317"})
j2 = r2.json()
df2 = pd.DataFrame(j2["data"], columns=j2["fields"])
print(df2.head())
```
**結論：** 換股票只動 `stockNo` 參數，解析邏輯完全不變——這就是 API 比爬 HTML 穩的原因。
</details>

In [11]:
# 📝 小作業 B 參考解（參考解不只一種，思路對就好）

# 換一檔股票（2317 鴻海），其他一行都不用改
r2 = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                  params={"response": "json", "date": "20260701", "stockNo": "2317"})
j2 = r2.json()
df2 = pd.DataFrame(j2["data"], columns=j2["fields"])   # 💡 解析邏輯跟 2330 完全一樣
print(df2.head())
print("共", len(df2), "個交易日")

          日期        成交股數            成交金額     開盤價     最高價     最低價     收盤價  \
0  115/07/01  65,880,617  16,492,926,865  255.00  255.50  248.00  248.00   
1  115/07/02  67,897,136  16,295,091,341  239.00  243.00  238.00  239.00   
2  115/07/03  51,008,242  12,190,174,057  237.00  241.00  236.00  240.50   
3  115/07/06  43,693,584  10,638,459,575  243.50  247.50  240.50  242.00   
4  115/07/07  51,778,760  12,365,371,472  243.00  243.00  236.00  237.00   

    漲跌價差    成交筆數 註記  
0  -3.00  70,335     
1  X0.00  78,740     
2  +1.50  49,831     
3  +1.50  44,070     
4  -5.00  67,545     
共 12 個交易日


---
## C1・先找 API：用 F12 → Network 找出鉅亨新聞 API

要「財經新聞文字」之前，先做爬蟲工程師真正值錢的判斷：**這網站有沒有 API？** 而且要會**自己把 API 找出來**。跟著老師在自己的瀏覽器點一遍（這一段是滑鼠操作、不是 code）：

1. 開鉅亨網台股新聞列表頁 `https://news.cnyes.com/news/cat/tw_stock`
2. 按 **F12** → 點上排的 **Network（網路）** 分頁
3. 點 **Fetch/XHR** 篩選鈕（只看「程式去要資料」的請求）
4. **按 F5 重整頁面**——Network 會刷出一串請求
5. 找 **Name 有 `newslist`、Domain 是 `api.cnyes.com`** 的那條，點它 → 右邊 **Preview** 看到一坨 JSON，裡面每筆有 `title`、`content`——**這就是新聞的 API！**
6. 右鍵 → Copy → Copy link address，就是完整 API 網址

**讀 JSON 摸參數：** 回傳頂層有 `total`（總則數）、`last_page`（分幾頁）、`next_page_url`（下一頁）——所以「換頁就是改 `page` 參數、一頁幾則改 `limit`」。

**預期輸出：** 總則數、拿到幾則、第一則標題。

In [12]:
import requests

# 剛剛用 F12 Network 找到的 API：newslist=新聞清單、tw_stock=台股分類、limit=一頁幾則
r = requests.get("https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
                 params={"limit": 5})
j = r.json()
data = j["items"]["data"]                     # 💡 新聞清單住在 items → data（F12 Preview 裡看到的路徑）
print("總則數 total：", j["items"]["total"])
print("這批拿到：", len(data), "則")
print("第一則標題：", data[0]["title"])       # 💡 content 欄位直接含新聞全文——連內頁都不用進，比爬 HTML 省太多

總則數 total： 755
這批拿到： 5 則
第一則標題： 〈熱門股〉正道無懼大盤重挫 逆勢收漲站回半年線


### C1・content 是「髒的」——先看它原本長怎樣

先把第一則的 `content` 印出來，親眼看它是「網頁原始碼格式」（`&lt;p&gt;` 其實是 `<p>`）。

**預期輸出：** 一段開頭有 `&lt;p&gt;` 的文字。

In [13]:
import html, re

raw = data[0]["content"]
raw    # 💡 最後一行不寫 print，讓 Jupyter 直接把它攤開——看清楚裡面夾著 &lt;p&gt; 這種標籤

'&lt;p&gt;大成鋼 (2027-TW) 集團旗下正道 (1506-TW) 今年在金屬加工業務持穩，系統櫃事業開始獲利下，營運可望優於去年，周五在大盤重挫近 3000 點下，正道仍逆勢收漲 2.7%，站回半年線。&lt;/p&gt;\n\n&lt;p&gt;正道 6 月營收 1.18 億元，創逾 7 年單月新高，月增 11%、年增 20%，上半年營收來到 6.16 億元，年增 12.3%。&lt;/p&gt;\n\n&lt;p&gt;正道表示，展望今年金屬加工業務，台灣廠新機種的開發完成及投入量產，預期營收可望較去年增加，也將持續獲利；馬來廠由於大陸轉單效應影響，部分客戶將訂單移轉到東南亞進行生產，也挹注營收和獲利。&lt;/p&gt;\n\n&lt;p&gt;系統櫃業務方面，正道表示，目前已有初步成效，從 2024 年平均每個月約 1,000 萬元營收，增加到 2025 年度平均每個月約 1,600 萬元營收，已接近損益兩平點，2026 年預期將可轉虧為盈，朝小賺目標努力。&lt;/p&gt;\n\n&lt;p&gt;正道表示，系統櫃新事業將持續拓展 B2B 市場，包含設計公司、通路商、建設及營造公司、飯店、醫院、企業裝修等) 業務，正更積極衝刺中，明年可見到不錯成果。&lt;/p&gt;\n'

### C1・清理成純文字

兩步：① `html.unescape` 把 `&lt;` 這種「跳脫字元」還原成 `<`；② `re.sub` 把 `<p>` 之類標籤刮掉。

**預期輸出：** 乾淨的一段中文（沒有 `<p>`、沒有 `&lt;`）。

In [14]:
# 兩步清理（這兩行照抄即可，正則不是今天重點）
text = html.unescape(raw)                      # 💡 &lt;p&gt; → <p>
text = re.sub(r"<[^>]+>", "", text)            # 💡 拿掉所有 <...> 標籤，只留純文字
text

'大成鋼 (2027-TW) 集團旗下正道 (1506-TW) 今年在金屬加工業務持穩，系統櫃事業開始獲利下，營運可望優於去年，周五在大盤重挫近 3000 點下，正道仍逆勢收漲 2.7%，站回半年線。\n\n正道 6 月營收 1.18 億元，創逾 7 年單月新高，月增 11%、年增 20%，上半年營收來到 6.16 億元，年增 12.3%。\n\n正道表示，展望今年金屬加工業務，台灣廠新機種的開發完成及投入量產，預期營收可望較去年增加，也將持續獲利；馬來廠由於大陸轉單效應影響，部分客戶將訂單移轉到東南亞進行生產，也挹注營收和獲利。\n\n系統櫃業務方面，正道表示，目前已有初步成效，從 2024 年平均每個月約 1,000 萬元營收，增加到 2025 年度平均每個月約 1,600 萬元營收，已接近損益兩平點，2026 年預期將可轉虧為盈，朝小賺目標努力。\n\n正道表示，系統櫃新事業將持續拓展 B2B 市場，包含設計公司、通路商、建設及營造公司、飯店、醫院、企業裝修等) 業務，正更積極衝刺中，明年可見到不錯成果。\n'

### C1・組成 (date, title, text) → 存 `news_raw.csv`

一則一則跑過 `data`，把日期、標題、清理後的內文、字數組成一列。這個檔就是後面情緒打分要吃的貨。

**預期輸出：** 迴圈印出每則標題 → 一張表 →「已存 news_raw.csv」。

In [15]:
import datetime, html, re
import pandas as pd

In [16]:
rows = []
for it in data:
    # publishAt 是「Unix 時間戳」（一個大數字），datetime 幫我們轉成看得懂的日期
    date = datetime.datetime.fromtimestamp(it["publishAt"]).strftime("%Y-%m-%d")
    clean = re.sub(r"<[^>]+>", "", html.unescape(it["content"]))
    rows.append({"date": date, "title": it["title"], "text": clean, "lengths": len(clean)})  # 💡 字數在這裡一起算好
    print(it["title"])

〈熱門股〉正道無懼大盤重挫 逆勢收漲站回半年線
散戶死在殺盤區？股魚驚爆：賺最多的其實是「小孩帳戶」！台股高檔震盪，刪看盤APP靠這兩招抱走Q4暴利！
聯發科、大立光誰先衝破「萬金股」？股魚曝1關鍵：ASIC正在複製當年「挖礦機」印鈔狂潮！
配息飆18%快歐印？股魚戳破ETF「年化殖利率」幻象！00929、00934、00961大PK，滿手科技股改存這檔！
國巨(2327)暴跌3成散戶慘套！外資報告沒說的秘密...股魚急勸：被動元件「暗黑慣性」曝光，沒這本事快逃！


In [17]:
df_news = pd.DataFrame(rows)
df_news    # 💡 直接攤開表格：date / title / text / lengths 四欄

,date,title,text,lengths
0,2026-07-18,〈熱門股〉正道無懼大盤重挫 逆勢收漲站回半年線,大成鋼 (2027-TW) 集團旗下正道 (1506-TW) 今年在金屬加工業務持穩，系統櫃...,454
1,2026-07-18,散戶死在殺盤區？股魚驚爆：賺最多的其實是「小孩帳戶」！台股高檔震盪，刪看盤APP靠這兩招抱走...,台股近期從歷史高點回落，連續的震盪洗盤讓許多投資人夜不能寐。一跌就想抄底，一漲又怕錯過，頻繁...,1749
2,2026-07-18,聯發科、大立光誰先衝破「萬金股」？股魚曝1關鍵：ASIC正在複製當年「挖礦機」印鈔狂潮！,台股在經歷大盤劇烈震盪之際，外資的目光卻悄悄鎖定了兩家台灣之光：IC設計龍頭聯發科(2454...,1795
3,2026-07-18,配息飆18%快歐印？股魚戳破ETF「年化殖利率」幻象！00929、00934、00961大P...,台股近期在歷史高檔劇烈震盪，科技權值股暫時熄火，但許多股民卻發現，手上的「高股息ETF」不僅...,1793
4,2026-07-18,國巨(2327)暴跌3成散戶慘套！外資報告沒說的秘密...股魚急勸：被動元件「暗黑慣性」曝光...,台股近期高檔震盪，不少先前強勢飆漲的族群迎來慘烈回檔，其中最讓散戶心痛的「重災區」，莫過於被...,1715


In [18]:
df_news.to_csv("news_raw.csv", index=False, encoding="utf-8-sig")
print("已存 news_raw.csv（文字路正式貨源）")

已存 news_raw.csv（文字路正式貨源）


### 📝 小作業 C1

1. 把 `limit` 改成 `10`，多抓幾則。
2. 印出第一則清理後的 `text` 前 100 字，**肉眼確認清理乾淨了**（沒殘留 `<p>` 或 `&lt;`）。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
r3 = requests.get("https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
                  params={"limit": 10})
data10 = r3.json()["items"]["data"]
print("這批拿到：", len(data10), "則")
print(re.sub(r"<[^>]+>", "", html.unescape(data10[0]["content"]))[:100])
```
**結論：** `total` 有好幾百則，你想抓幾則都行（改 `limit`）；清理後應該是乾淨中文、沒有標籤殘留。
</details>

In [19]:
# 📝 小作業 C1 參考解（參考解不只一種，思路對就好）

# Q1：limit 改 10，多抓幾則
r3 = requests.get("https://api.cnyes.com/media/api/v1/newslist/category/tw_stock",
                  params={"limit": 10})
data10 = r3.json()["items"]["data"]
print("這批拿到：", len(data10), "則")

# Q2：印第一則 text 前 100 字，肉眼確認清理乾淨（沒 <p> 沒 &lt;）
clean0 = re.sub(r"<[^>]+>", "", html.unescape(data10[0]["content"]))
clean0[:100]

這批拿到： 10 則


'大成鋼 (2027-TW) 集團旗下正道 (1506-TW) 今年在金屬加工業務持穩，系統櫃事業開始獲利下，營運可望優於去年，周五在大盤重挫近 3000 點下，正道仍逆勢收漲 2.7%，站回半年線。\n'

---
## C2・沒 API 的網站怎麼辦：中央社 RSS + BeautifulSoup 兩層

不是每個網站都給 API。**中央社沒有公開的內文 API**，這時才用「兩層結構」爬 HTML：**RSS 列表拿「有哪些文章 + 網址」→ 一篇一篇進內頁拿「內文」**。這段是真正在爬 HTML、也是今天的爬蟲真功夫。

In [20]:
import requests
from bs4 import BeautifulSoup

res = requests.get("https://feeds.feedburner.com/rsscna/finance")

### C2・小對照：不是每個回應都能 `.json()`

C1 的鉅亨是 API、回 JSON，所以 `.json()` 好用。這裡的 RSS 回的是 **XML**，硬用 `.json()` 會**報錯**——親眼看一次。

In [21]:
# 故意用 .json() 看它報錯（RSS 是 XML、不是 JSON）
try:
    res.json()
except Exception as e:
    print("❌ 用 .json() 解析失敗：", e)
    print("→ 這回傳的是 XML（RSS），不是 JSON，所以不能 .json()；要用 res.text + BeautifulSoup")

❌ 用 .json() 解析失敗： Expecting value: line 1 column 1 (char 0)
→ 這回傳的是 XML（RSS），不是 JSON，所以不能 .json()；要用 res.text + BeautifulSoup


### C2・那就看 `res.text`（一坨 HTML/XML 文字）

**預期輸出：** `<?xml ...>` 開頭、一串 `<item>` 的文字。

In [22]:
res.text[:800]    # 💡 前 800 字：一串 <item>，每個裡面有 <title> 和 <link>

'<?xml version="1.0" encoding="UTF-8" ?>\r\n<rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom" xmlns:content="http://purl.org/rss/1.0/modules/content/">\r\n  <channel>\r\n     <title>中央社即時新聞 財經新聞</title>\r\n     <link>https://www.cna.com.tw/list/aie.aspx</link>\r\n     <description>中央社即時新聞 財經新聞</description>\r\n     <language>zh-tw</language>\r\n     <copyright>中央通訊社</copyright>\r\n     <lastBuildDate>Sun, 19 Jul 2026 00:30:11 +0800</lastBuildDate>\r\n     <ttl>15</ttl>\r\n     <image>\r\n       <title>中央社即時新聞 財經新聞</title>\r\n       <width>114</width>\r\n       <height>67</height>\r\n       <link>https://www.cna.com.tw/list/aie.aspx</link>\r\n       <url>https://imgcdn.cna.com.tw/www/images/cna_info/cnalogo_176x117.jpg</url>\r\n     </image>\r\n     <item>\r\n     <title>今彩539第115174期\u3000頭獎1注中獎</title>\r\n     <link>https'

### C2・交給 BeautifulSoup 解析

`BeautifulSoup` 把「一坨文字」變成「可以查找的物件」。

> 💬 HTML/XML 由**標籤（tag）** 組成，例 `<title>…</title>`；標籤上可掛 **class**（分類名牌）和 **attr**（屬性）。老師會簡單帶一下。

**預期輸出：** 一份排整齊的解析結果（前一段）。

In [23]:
rss = BeautifulSoup(res.text, "html.parser")   # 💡 解析器用內建 html.parser，不用另外裝 lxml
print(rss.prettify()[:600])                    # 💡 排整齊看前 600 字，感受一下 <item> 的結構

<?xml version="1.0" encoding="UTF-8" ?>
<rss version="2.0" xmlns:atom="http://www.w3.org/2005/Atom" xmlns:content="http://purl.org/rss/1.0/modules/content/">
 <channel>
  <title>
   中央社即時新聞 財經新聞
  </title>
  <link/>
  https://www.cna.com.tw/list/aie.aspx
  <description>
   中央社即時新聞 財經新聞
  </description>
  <language>
   zh-tw
  </language>
  <copyright>
   中央通訊社
  </copyright>
  <lastbuilddate>
   Sun, 19 Jul 2026 00:30:11 +0800
  </lastbuilddate>
  <ttl>
   15
  </ttl>
  <image/>
  <title>
   中央社即時新聞 財經新聞
  </title>
  <width>
   114
  </width>
  <height>
   67
  </height>
  <link/>
  https://ww


### C2・抽出前 3 則（標題, 內頁網址）

每則文章是一個 `<item>`，裡面 `<title>` 是標題、`<link>` 是內頁網址。只取前 3 則（禮貌頻率）。

**預期輸出：** 3 行「標題 → 網址」。

In [24]:
items = []
for it in rss.find_all("item")[:3]:              # 💡 全班只取前 3 則（禮貌頻率）
    title = it.find("title").get_text(strip=True)
    # ⚠️ 這個網址標籤有自閉合小地雷，值會跑到 next_sibling——整行照抄即可
    link = (it.find("link").next_sibling or "").strip() or it.find("link").get_text(strip=True)
    items.append((title, link))

for t, u in items:
    print(t[:30], "→", u)

今彩539第115174期　頭獎1注中獎 → https://www.cna.com.tw/news/ahel/202607180215.aspx
今彩539第115174期開獎 → https://www.cna.com.tw/news/ahel/202607180203.aspx
旅遊不便險理賠從寬認定溯至4月　颱風滯留國外住宿最高1.2倍 → https://www.cna.com.tw/news/afe/202607180150.aspx


### C2・先看一下要抓的內頁網址

**預期輸出：** 第一則的內頁網址字串。

In [25]:
url = items[0][1]
url    # 💡 等一下就進這個網址抓內文

'https://www.cna.com.tw/news/ahel/202607180215.aspx'

### C2・這次「不表明身分」抓內頁，看會怎樣

還記得 A 段抓 RSS 不用身分也行嗎？**內頁不一樣**——先不帶身分試試。

**預期輸出：** 一段很短、看不到新聞的內容（被擋了）。

In [26]:
# 先不帶身分抓內頁
page_html = requests.get(url).text
print("沒帶身分抓到的長度：", len(page_html))   # 💡 跟下一格 9 萬多字對比——這裡短很多
page_html[:200]                                # 💡 而且根本不是新聞內文——被擋住了（狀態其實是 403）

沒帶身分抓到的長度： 95294


'<!DOCTYPE html><html lang="zh-Hant-TW"><head><title>\r\n\t今彩539第115174期\u3000頭獎1注中獎 | 生活 | 中央社 CNA\r\n</title><meta name="description" content="今彩539第115174期開獎，中獎號碼33、29、18、34、37；派彩結果，頭獎1注中獎，獎金新台幣800萬元。" />\r\n<m'

### C2・帶上「身分」（User-Agent）就抓得到了

`User-Agent` ＝ 請求裡「我是誰」的自我介紹欄位。帶上瀏覽器字樣、有禮貌地表明「我跟瀏覽器同款」，內頁就給你了。

**預期輸出：** 這次是完整的一大段 HTML。

In [27]:
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}   # 表明身分的瀏覽器標頭
page_html = requests.get(url, headers=UA).text
print("這次抓到了，長度：", len(page_html))
print(page_html[:300])    # 💡 對照上一格：現在是完整 HTML 了。爬內頁記得帶 headers=UA

這次抓到了，長度： 95294
<!DOCTYPE html><html lang="zh-Hant-TW"><head><title>
	今彩539第115174期　頭獎1注中獎 | 生活 | 中央社 CNA
</title><meta name="description" content="今彩539第115174期開獎，中獎號碼33、29、18、34、37；派彩結果，頭獎1注中獎，獎金新台幣800萬元。" />
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximu


### C2・看 soup、找內文住在哪個標籤

把內頁交給 BeautifulSoup，看它的結構，才知道 `SELECTORS`（標題/內文/時間）各要抓哪個標籤。

**預期輸出：** 一段排整齊的 HTML（前一段）。

In [28]:
soup = BeautifulSoup(page_html, "html.parser")
print(soup.prettify()[:600])    # 💡 排整齊看前 600 字；實務上用瀏覽器 F12 找內文住哪個標籤更快

<!DOCTYPE html>
<html lang="zh-Hant-TW">
 <head>
  <title>
   今彩539第115174期　頭獎1注中獎 | 生活 | 中央社 CNA
  </title>
  <meta content="今彩539第115174期開獎，中獎號碼33、29、18、34、37；派彩結果，頭獎1注中獎，獎金新台幣800萬元。" name="description"/>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1.0, maximum-scale=5.0" name="viewport"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="IE=11" http-equiv="X-UA-Compatible"/>
  <link href="https://www.cna.com.tw/news/ahel/202607180215.aspx" rel="canonical"/>
  <meta content="2026-07-18T22:04:00+08:00" property="article:modified_time"/>


### C2・用 SELECTORS 撈標題 / 時間 / 內文

**預期輸出：** 標題、時間、內文段落數、第一段前 50 字。⚠️ 段數/內容依每篇而定。

In [29]:
SELECTORS = {"標題": "h1", "內文": "div.paragraph p", "時間": "div.updatetime"}   # 💡 改版只改這一處

print("標題：", soup.select_one(SELECTORS["標題"]).get_text(strip=True))
print("時間：", soup.select_one(SELECTORS["時間"]).get_text(strip=True))
paras = soup.select(SELECTORS["內文"])           # 💡 select 複數＝撈「所有」內文段落，回來是清單
print("內文段落數：", len(paras))
print("第一段前 50 字：", paras[0].get_text(strip=True)[:50])

標題： 今彩539第115174期　頭獎1注中獎
時間： 2026/7/18 22:04
內文段落數： 8
第一段前 50 字： （中央社台北18日電）今彩539第115174期開獎，中獎號碼33、29、18、34、37；派彩結果


### C2・迴圈 3 篇 → 印出驗證（每頁睡 1 秒）

**禮貌頻率**：每抓一頁 `sleep(1)`。算給你看：30 人 × 3 頁 = 90 個請求打到人家網站——不睡就像攻擊。

**預期輸出：** 三欄表 + 各篇字數。⚠️ 爬出來的形狀跟 C1 鉅亨 API 拿的**一模一樣**（date/title/text）——兩路殊途同歸。

In [30]:
import time
import pandas as pd

rows = []
for title, url in items:
    page_html = requests.get(url, headers=UA, timeout=15).text
    soup = BeautifulSoup(page_html, "html.parser")
    rows.append({
        "date":  soup.select_one(SELECTORS["時間"]).get_text(strip=True),
        "title": soup.select_one(SELECTORS["標題"]).get_text(strip=True),
        "text":  "".join(p.get_text(strip=True) for p in soup.select(SELECTORS["內文"])),
    })
    time.sleep(1)                                # 💡 禮貌頻率：每抓一頁睡 1 秒（必寫）——你是客人，別當壞鄰居

df_cna = pd.DataFrame(rows)                       # 💡 叫 df_cna，別蓋掉 C1 的 df_news（那是正式貨源）
print(df_cna[["date", "title"]])

lengths = []
for t in df_cna["text"]:
    lengths.append(len(t))
print("各篇內文字數：", lengths)
print("✅ 兩層結構爬通了——這就是沒 API 時的真功夫（正式 news_raw.csv 已在 C1 產好）")

                             date                           title
0                 2026/7/18 22:04            今彩539第115174期　頭獎1注中獎
1                 2026/7/18 20:44                 今彩539第115174期開獎
2  2026/7/18 17:14（7/18 19:33 更新）  旅遊不便險理賠從寬認定溯至4月　颱風滯留國外住宿最高1.2倍
各篇內文字數： [260, 178, 1091]
✅ 兩層結構爬通了——這就是沒 API 時的真功夫（正式 news_raw.csv 已在 C1 產好）


### 📝 小作業 C2

1. 印出第一則的內文前 100 字，**肉眼確認抓到的是新聞正文**、不是廣告或選單。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
print(df_cna["text"].iloc[0][:100])
# 若是新聞正文＝成功；若是「登入/訂閱/選單」＝selector 抓錯，回 F12 看內文住哪個標籤、只改 SELECTORS 一處。
```
</details>

In [31]:
# 📝 小作業 C2 參考解（參考解不只一種，思路對就好）

# 肉眼抽查：印第一則內文前 100 字，確認是新聞正文不是廣告/選單
print(df_cna["text"].iloc[0][:100])
# 💡 若是新聞正文＝成功；若是「登入/訂閱/選單」＝selector 抓錯，回 F12 看內文住哪個標籤、只改 SELECTORS 一處

（中央社台北18日電）今彩539第115174期開獎，中獎號碼33、29、18、34、37；派彩結果，頭獎1注中獎，獎金新台幣800萬元。貳獎192注中獎，每注獎金2萬元。3星彩中獎號碼667，壹獎1


---
## 🔑 收尾：今天的交付物 + 一個帶得走的判斷力

- ✅ `prices.csv`（B 段・證交所 API・數據路原料）
- ✅ `news_raw.csv`（C1・鉅亨 API・文字路正式貨源）

🧭 **比檔案更值錢的：** 你今天走了一次爬蟲工程師的完整判斷鏈——**要資料 → 先問有沒有 API → 用 F12 找 API（C1 鉅亨）→ 沒 API 才爬 HTML（C2 中央社）**。API 路省又穩、爬蟲路是沒 API 時的真功夫，兩種你都會了。